# x402 `batch-settlement` scheme — facilitator test

Tests the **`batch-settlement`** scheme end-to-end against a running facilitator: open a
channel with an escrowed deposit, accumulate several requests as off-chain vouchers, then
claim them all in a single on-chain transaction.

**Where this fits among the sibling notebooks:**
- **This notebook** — `batch-settlement` scheme.
- [`x402_exact.ipynb`](./x402_exact.ipynb) — same idea for the `exact` scheme (one
  on-chain transfer per request, no channel).
- [`genimg_x402_buyer.ipynb`](./genimg_x402_buyer.ipynb) — the buyer's perspective
  instead: a real production integration paid via `@x402/fetch`, no direct facilitator calls.

## Which side is this notebook playing?

Both — batch-settlement needs both halves to demonstrate anything, so the sections are
labelled by role rather than split across files:

| Sections | Role | In production this is… |
|---|---|---|
| 2–4, 8 (deposit, vouchers) | **Buyer** — signs the deposit and each cumulative voucher | `website/` (browser wallet) |
| 8b–9 (approve, claim) | **Seller** — approves the fee allowance, authorizes and submits the claim | `scw_js/` + its claim/settle cron |
| 1, 5, 7 (`/supported`, `/verify`, `/settle`) | Calls **into the facilitator** | `x402_facilitator/` — this package |

The facilitator is neither buyer nor seller: it verifies signatures and submits the
deposit/claim/settle transactions on-chain, and never holds funds.

## Prerequisites

- Deno Jupyter kernel (same as the sibling `.ipynb` notebooks — see this directory's README).
- The package's single **`x402_facilitator/.env`** (one level up — there is no per-notebook
  `.env`) with: `TEST_WALLET_PRIVATE_KEY` (buyer; funded on **Base Sepolia** — ETH for gas
  plus testnet USDC via https://faucet.circle.com/), `NFT_WALLET_PUBLIC_KEY` (seller /
  recipient), and `FACILITATOR_WALLET_PUBLIC_KEY`. `NFT_WALLET_PRIVATE_KEY` is also needed
  for section 8b (the seller approves the facilitator's fee allowance) — optional if you
  only run through section 8.
- A facilitator running locally: from the parent dir `npm install && npm run build && npm run dev`
  → `http://localhost:8080`.

**Why Base Sepolia:** the canonical `BATCH_SETTLEMENT_ADDRESS` (`0x4020…0003`) is deployed on
Base Sepolia, Optimism mainnet and Base mainnet — but **not** Optimism Sepolia. So testnet
runs use Base Sepolia.

> ⚠️ The cells that hit `/settle` perform **real on-chain** actions (escrowing USDC, claiming
> it). On mainnet they are additionally gated behind `CONFIRM_MAINNET_WRITES`.

In [1]:
// Setup: imports + config
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount, generatePrivateKey } from "npm:viem@2/accounts";
import { createPublicClient, http, formatUnits } from "npm:viem@2";
import { baseSepolia, base, optimism } from "npm:viem@2/chains";

// All env lives in the package's single .env, one level up: x402_facilitator/.env
// (TEST_WALLET_PRIVATE_KEY, NFT_WALLET_PUBLIC_KEY, FACILITATOR_WALLET_*).
// `examplePath: null` disables dotenv's default assertion that every key in a
// ./.env.example must also be present — we're not using a per-notebook .env.
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
if (!PRIVATE_KEY) {
    throw new Error("TEST_WALLET_PRIVATE_KEY missing from x402_facilitator/.env — " +
        "add a key for a wallet funded on the selected network.");
}
const PAY_TO_ADDRESS = env.NFT_WALLET_PUBLIC_KEY ?? "0x209693Bc6afc0C5328bA36FaF03C514EF312287C";
const FACILITATOR_ADDRESS = env.FACILITATOR_WALLET_PUBLIC_KEY; // from ../.env — used for the gas check
const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);

// Needed for section 8b: the merchant (recipient) approves the facilitator's fee token
// allowance for claim/settle (Phase 3). Optional — sections 1-8 don't need it.
const RECIPIENT_PRIVATE_KEY = env.NFT_WALLET_PRIVATE_KEY;
const recipientAccount = RECIPIENT_PRIVATE_KEY
    ? privateKeyToAccount(`0x${RECIPIENT_PRIVATE_KEY.replace(/^0x/, "")}`)
    : null;

// The receiver self-manages an authorizer key (signs claims/refunds). In production
// the seller holds this and advertises its address in the 402's extra.receiverAuthorizer.
// Here we generate a throwaway one for the run.
const receiverAuthorizer = privateKeyToAccount(generatePrivateKey());

console.log("🚀 x402 batch-settlement — facilitator test");
console.log(`   Payer (buyer)     : ${account.address}`);
console.log(`   Recipient         : ${PAY_TO_ADDRESS}`);
console.log(`   receiverAuthorizer: ${receiverAuthorizer.address}`);
if (recipientAccount && recipientAccount.address.toLowerCase() !== PAY_TO_ADDRESS.toLowerCase()) {
    console.warn("⚠️  NFT_WALLET_PRIVATE_KEY does not match PAY_TO_ADDRESS — section 8b's approve() would sign for the wrong address.");
}

🚀 x402 batch-settlement — facilitator test
   Payer (buyer)     : 0x553179556FC2A39e535D65b921e01fA995E79101
   Recipient         : 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C
   receiverAuthorizer: 0x93eafeeD33503966A4938Ebe35FE474377973B0D


## Network selection

Modeled on `x402_exact.ipynb`, but **Optimism Sepolia is disabled** — the canonical
`BATCH_SETTLEMENT_ADDRESS` (`0x4020…0003`) is **not** deployed there. Valid networks (all have the contract):
Base Sepolia (default), Base Mainnet, Optimism Mainnet.

| Chain | Testnet | Mainnet |
|---|---|---|
| Base | `eip155:84532` ✅ | `eip155:8453` |
| Optimism | ~~`eip155:11155420`~~ (no contract) | `eip155:10` |

**Token:** set `TOKEN` to `"EURC"` (default) or `"USDC"`. EURC exists on Base only.


In [2]:
// ── Network selection ──────────────────────────────────────────────
const USE_MAINNET = false;  // ⚠️ true = REAL MONEY. Keep false unless you mean it.

// Second, deliberate opt-in for the two cells that actually move money on mainnet
// (settle + claim). USE_MAINNET alone only selects the network; this one arms the
// writes. Left false, those cells skip with a notice instead of spending anything.
const CONFIRM_MAINNET_WRITES = false;
const USE_BASE = true;      // Base by default. (Optimism + testnet is unavailable.)

// Guard: Optimism Sepolia has no batch-settlement contract.
if (!USE_BASE && !USE_MAINNET) {
    throw new Error("Optimism Sepolia is unsupported for batch-settlement (no canonical contract). " +
        "Use Base Sepolia (USE_BASE=true) or a mainnet (USE_MAINNET=true).");
}

const NETWORK_CONFIG = {
    "base-testnet": {
        chain: baseSepolia, chainId: 84532, caip2Network: "eip155:84532" as const,
        networkName: "Base Sepolia (Testnet)", usdcName: "USDC",
        usdcAddress: "0x036CbD53842c5426634e7929541eC2318f3dCF7e" as `0x${string}`,
        rpcUrl: "https://sepolia.base.org", explorer: "https://sepolia.basescan.org",
        faucet: "https://faucet.circle.com/",
    },
    "base-mainnet": {
        chain: base, chainId: 8453, caip2Network: "eip155:8453" as const,
        networkName: "Base Mainnet", usdcName: "USD Coin",
        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913" as `0x${string}`,
        rpcUrl: "https://mainnet.base.org", explorer: "https://basescan.org",
        faucet: "Bridge: https://bridge.base.org",
    },
    "optimism-mainnet": {
        chain: optimism, chainId: 10, caip2Network: "eip155:10" as const,
        networkName: "Optimism Mainnet", usdcName: "USD Coin",
        usdcAddress: "0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85" as `0x${string}`,
        rpcUrl: "https://mainnet.optimism.io", explorer: "https://optimistic.etherscan.io",
        faucet: "Bridge: https://app.optimism.io/bridge",
    },
};
const configKey = USE_MAINNET ? (USE_BASE ? "base-mainnet" : "optimism-mainnet") : "base-testnet";
const config = NETWORK_CONFIG[configKey];

// ── Token selection ────────────────────────────────────────────────
// EURC exists on Base only (Circle has no Optimism deployment). A EURC channel is a separate
// channel from a USDC one — the token is part of the channel id — so it needs its own deposit.
const TOKEN: "USDC" | "EURC" = "EURC";
// Address and EIP-712 domain name read on-chain (name() = "EURC", version "2").
const EURC_BY_NETWORK: Record<string, { address: `0x${string}`; name: string }> = {
    "eip155:84532": { address: "0x808456652fdb597867f38412077A9182bf77359F", name: "EURC" }, // Base Sepolia
    "eip155:8453": { address: "0x60a3E35Cc302bFA44Cb288Bc5a4F316Fdb1adb42", name: "EURC" },  // Base
};
const eurc = EURC_BY_NETWORK[config.caip2Network];
if (TOKEN === "EURC" && !eurc) {
    throw new Error(`EURC is not deployed on ${config.networkName} — set USE_BASE = true, or TOKEN = "USDC".`);
}
// The token the payer escrows, and so the one the facilitator charges its claim fee in.
const tokenAddress = TOKEN === "EURC" ? eurc!.address : config.usdcAddress;
const tokenName = TOKEN === "EURC" ? eurc!.name : config.usdcName;

// Max per-request price (6 decimals — USDC and EURC alike). Channel deposit ≈ depositMultiplier(5) × this.
const MAX_PRICE = "4000"; // $0.004  → deposit ≈ $0.02

const FACILITATOR_URL = "http://localhost:8080"; // or "https://facilitator.fretchen.eu"
// const FACILITATOR_URL = "https://facilitator.fretchen.eu"
const VERIFY_URL = `${FACILITATOR_URL}/verify`;
const SETTLE_URL = `${FACILITATOR_URL}/settle`;
const SUPPORTED_URL = `${FACILITATOR_URL}/supported`;

console.log(USE_MAINNET ? `🚨 REAL MONEY on ${config.networkName}` : `🧪 ${config.networkName}`);
console.log(`   ${config.caip2Network}  •  ${TOKEN} (${tokenName}) @ ${tokenAddress}`);
console.log(`   Facilitator: ${FACILITATOR_URL}`);

🧪 Base Sepolia (Testnet)
   eip155:84532  •  EURC (EURC) @ 0x808456652fdb597867f38412077A9182bf77359F
   Facilitator: http://localhost:8080


## 1. `/supported` — assert the facilitator advertises `batch-settlement`

This is the core Phase A check: after registering `BatchSettlementEvmScheme` in `facilitator_instance.ts`,
the facilitator's `/supported` should list the `batch-settlement` scheme for each network.


In [3]:
const supported = await (await fetch(SUPPORTED_URL)).json();
const kinds = supported.kinds ?? [];
const batchKinds = kinds.filter((k: any) => k.scheme === "batch-settlement");
const exactKinds = kinds.filter((k: any) => k.scheme === "exact");

console.log(`schemes advertised: exact=${exactKinds.length}, batch-settlement=${batchKinds.length}`);
console.log("batch-settlement kinds:", JSON.stringify(batchKinds, null, 2));

if (batchKinds.some((k: any) => k.network === config.caip2Network)) {
    console.log(`✅ batch-settlement advertised for ${config.caip2Network}`);
} else {
    console.log(`❌ batch-settlement NOT advertised for ${config.caip2Network} — check facilitator registration`);
}
// With no receiverAuthorizer configured on the facilitator (self-managed receiver),
// `extra.receiverAuthorizer` should be absent here — the seller supplies its own.

schemes advertised: exact=4, batch-settlement=3
batch-settlement kinds: [
  {
    "x402Version": 2,
    "scheme": "batch-settlement",
    "network": "eip155:10"
  },
  {
    "x402Version": 2,
    "scheme": "batch-settlement",
    "network": "eip155:8453"
  },
  {
    "x402Version": 2,
    "scheme": "batch-settlement",
    "network": "eip155:84532"
  }
]
✅ batch-settlement advertised for eip155:84532


## 2. Buyer setup

Constructs the **buyer**-side scheme — SDK class `BatchSettlementEvmScheme`, imported from
`.../batch-settlement/client` (the SDK's "client" = the paying wallet). In production this is
`website/`, signing with the connected browser wallet; here it's the test key from cell 1.

Two things differ from the `exact` scheme:
- There is **no `registerBatchSettlementEvmScheme` helper** — construct the scheme and register
  it on the `x402Client` manually.
- The scheme needs **channel storage**. A browser buyer would back this with `localStorage` so
  channel state survives a reload; in-memory is fine here since the notebook runs start to finish.

In [4]:
import { x402Client } from "npm:@x402/fetch@^2.17.0";
// SDK naming: "client" here means the buyer/payer role (→ website/ in production),
// NOT scw_js (which plays the "server"/seller role — see the terminology table above).
import { BatchSettlementEvmScheme, InMemoryClientChannelStorage }
    from "npm:@x402/evm@^2.17.0/batch-settlement/client";

// Reused later (section 8) to sign additional off-chain vouchers on this same channel.
const buyerSigner = { address: account.address, signTypedData: (a: any) => account.signTypedData(a) };
const buyerStorage = new InMemoryClientChannelStorage();
const buyerScheme = new BatchSettlementEvmScheme(buyerSigner, { storage: buyerStorage });

const buyerClient = new x402Client();
buyerClient.register(config.caip2Network, buyerScheme);
// The SDK's default spend controls only accept tokens in its own registry (USDC), so
// allowlist the token explicitly — EURC would be rejected otherwise.
buyerClient.setSpendControls({ allowedAssets: [{ network: config.caip2Network, asset: tokenAddress }] });
console.log("✅ buyer registered for", config.caip2Network, "(payer:", account.address + ")");

✅ buyer registered for eip155:84532 (payer: 0x553179556FC2A39e535D65b921e01fA995E79101)


## 3. Build batch-settlement payment requirements

In production these come from the **server's 402 response** (the scw_js resource server injects these via the
server scheme's `enhancePaymentRequirements`). **Confirmed required in `extra`:** `receiverAuthorizer`
(non-zero — the client throws `Payment requirements must include a non-zero extra.receiverAuthorizer` without it)
and `withdrawDelay`, plus the usual EIP-712 `name`/`version`.


In [5]:
const paymentRequirements = {
    scheme: "batch-settlement",
    network: config.caip2Network,
    amount: MAX_PRICE,
    asset: tokenAddress,
    payTo: PAY_TO_ADDRESS,
    maxTimeoutSeconds: 3600,
    extra: {
        name: tokenName,
        version: "2",
        receiverAuthorizer: receiverAuthorizer.address, // server's self-managed authorizer in prod
        withdrawDelay: 86400,
    },
};

const paymentRequired = {
    x402Version: 2,
    accepts: [paymentRequirements],
    resource: { url: "https://example.com/llm", description: "batch-settlement test", mimeType: "application/json" },
    extensions: {},
};
console.log(JSON.stringify(paymentRequirements, null, 2));

{
  "scheme": "batch-settlement",
  "network": "eip155:84532",
  "amount": "4000",
  "asset": "0x808456652fdb597867f38412077A9182bf77359F",
  "payTo": "0xAAEBC1441323B8ad6Bdf6793A8428166b510239C",
  "maxTimeoutSeconds": 3600,
  "extra": {
    "name": "EURC",
    "version": "2",
    "receiverAuthorizer": "0x93eafeeD33503966A4938Ebe35FE474377973B0D",
    "withdrawDelay": 86400
  }
}


## 4. Buyer creates the deposit + first cumulative voucher payload

On the first request the buyer opens a channel: an EIP-3009 deposit into the batch-settlement contract plus
an initial cumulative voucher. `createPaymentPayload` handles the signing. **Confirmed payload shape:**
`payload.type == "deposit"` with `channelConfig` (payer, payerAuthorizer, receiver, receiverAuthorizer, token,
withdrawDelay, salt), `voucher` (channelId, maxClaimableAmount, signature), and `deposit`. The deposit is sized
by the deposit policy (default `depositMultiplier: 5`), so the payer needs ≥ 5×MAX_PRICE USDC to settle.


In [6]:
const paymentPayload = await buyerClient.createPaymentPayload(paymentRequired as any);
console.log("✅ payload created (deposit + voucher):");
console.log(JSON.stringify(paymentPayload, null, 2).slice(0, 1200));

✅ payload created (deposit + voucher):
{
  "x402Version": 2,
  "payload": {
    "type": "deposit",
    "channelConfig": {
      "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
      "payerAuthorizer": "0x553179556FC2A39e535D65b921e01fA995E79101",
      "receiver": "0xAAEBC1441323B8ad6Bdf6793A8428166b510239C",
      "receiverAuthorizer": "0x93eafeeD33503966A4938Ebe35FE474377973B0D",
      "token": "0x808456652fdb597867f38412077A9182bf77359F",
      "withdrawDelay": 86400,
      "salt": "0x0000000000000000000000000000000000000000000000000000000000000000"
    },
    "voucher": {
      "channelId": "0x7d85daf9f85f907304bda59018e1f5c7b49d1843a6908be6d71efde938e78969",
      "maxClaimableAmount": "4000",
      "signature": "0xe41222b0029b8124354d0f8660ec21f080f24e1f626404aa3af9611bb21552050390eb6b6bc788ec577e1176ae0cf38fd662e61a84ed0f1b9e82988b9a65ef311b"
    },
    "deposit": {
      "amount": "20000",
      "authorization": {
        "erc3009Authorization": {
          "validAfter":

## 5. `/verify` the payload


In [7]:
const verifyRes = await fetch(VERIFY_URL, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ paymentPayload, paymentRequirements }),
});
const verifyResult = await verifyRes.json();
console.log(`verify (${verifyRes.status}):`, JSON.stringify(verifyResult, null, 2));

verify (200): {
  "isValid": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "extra": {
    "channelId": "0x7d85daf9f85f907304bda59018e1f5c7b49d1843a6908be6d71efde938e78969",
    "balance": "0",
    "totalClaimed": "0",
    "withdrawRequestedAt": 0,
    "refundNonce": "0"
  }
}


## 6. Pre-settlement balance check (like the demo notebook)

Settlement escrows the deposit on-chain, so the **payer needs USDC** (≈ `depositMultiplier` × MAX_PRICE) and
the **facilitator wallet needs ETH** for gas. Check before spending.


In [8]:
const publicClient = createPublicClient({ chain: config.chain, transport: http(config.rpcUrl) });
const erc20 = [{ inputs: [{ name: "a", type: "address" }], name: "balanceOf",
    outputs: [{ name: "", type: "uint256" }], stateMutability: "view", type: "function" }] as const;

const payerBalance = await publicClient.readContract({
    address: tokenAddress, abi: erc20, functionName: "balanceOf", args: [account.address] });
const DEPOSIT_MULTIPLIER = 5n; // batch-settlement default
const needed = BigInt(MAX_PRICE) * DEPOSIT_MULTIPLIER;

console.log(`💵 Payer ${TOKEN}: ${formatUnits(payerBalance, 6)}  (deposit needs ≈ ${formatUnits(needed, 6)})`);
if (payerBalance < needed) {
    console.log(`   ⚠️ insufficient ${TOKEN} on ${config.networkName} — ${config.faucet}`);
} else {
    console.log(`   ✅ enough ${TOKEN} to open the channel`);
}

// The facilitator submits the deposit tx, so it needs ETH for gas (from ../.env).
if (FACILITATOR_ADDRESS) {
    const facEth = await publicClient.getBalance({ address: FACILITATOR_ADDRESS as `0x${string}` });
    console.log(`⛽ Facilitator ETH: ${formatUnits(facEth, 18)}  (${FACILITATOR_ADDRESS})`);
    if (facEth === 0n) console.log(`   ⚠️ facilitator has no ETH for gas on ${config.networkName}`);
} else {
    console.log("⛽ FACILITATOR_WALLET_PUBLIC_KEY not found in ../.env — skipping gas check");
}

💵 Payer EURC: 19.92  (deposit needs ≈ 0.02)
   ✅ enough EURC to open the channel
⛽ Facilitator ETH: 0.009964430097829149  (0x3F8d2Fb6fEA24E70155bC61471936F3c9C30c206)


## 7. `/settle` — submit the deposit on-chain

> ⚠️ Real on-chain tx: escrows USDC into the batch-settlement contract. The facilitator dispatches the deposit
> internally by payload type — no separate endpoint. `deposit` is **free** — no fee `transferFrom`. (As of
> Phase 3, batch-settlement's flat fee applies only to `claim`/`settle`, the payload types that actually
> pay out — see section 8b and section 9.)


In [9]:
const maySettle = !USE_MAINNET || CONFIRM_MAINNET_WRITES;
if (!maySettle) {
    console.warn("⏭️  Skipped — this settles REAL funds on mainnet.");
    console.warn("    Set CONFIRM_MAINNET_WRITES = true in the network-selection cell to arm it.");
}

if (maySettle) {
    const settleRes = await fetch(SETTLE_URL, {
        method: "POST", headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ paymentPayload, paymentRequirements }),
    });
    const settleResult = await settleRes.json();
    console.log(`settle (${settleRes.status}):`, JSON.stringify(settleResult, null, 2));
    if (settleResult.transaction) console.log(`🔗 ${config.explorer}/tx/${settleResult.transaction}`);
    // deposit is fee-free (Phase 3 only fee-gates claim/settle) — settleResult should carry NO facilitatorFees / fee.collected.
}

settle (200): {
  "success": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "transaction": "0xe045c2e581b6325a9504256cf4f044d74d2b5178965262d5f71a7de2e3778e5a",
  "network": "eip155:84532",
  "extra": {
    "channelState": {
      "channelId": "0x7d85daf9f85f907304bda59018e1f5c7b49d1843a6908be6d71efde938e78969",
      "balance": "20000",
      "totalClaimed": "0",
      "withdrawRequestedAt": 0,
      "refundNonce": "0"
    }
  }
}
🔗 https://sepolia.basescan.org/tx/0xe045c2e581b6325a9504256cf4f044d74d2b5178965262d5f71a7de2e3778e5a


## 8. Accumulate several requests in the same channel (off-chain, no on-chain action)

This is the actual batching mechanic: once a channel is open (section 4), each further "request" — a chat
message, an image generation, anything metered per-call — just signs a **new, larger cumulative** voucher for
the SAME `channelId`. No transaction, no gas, no facilitator round-trip required. `signVoucher` is a standalone
buyer-side helper (same `.../batch-settlement/client` subpath) for exactly this — it doesn't re-open the channel,
it just bumps the authorized max.


In [10]:
import { signVoucher } from "npm:@x402/evm@^2.17.0/batch-settlement/client";

const channelId = paymentPayload.payload.voucher.channelId;
const channelConfig = paymentPayload.payload.channelConfig;

// "Request #2" and "#3" on the SAME channel — purely off-chain signing, cumulative amount grows.
const voucher2 = await signVoucher(buyerSigner, channelId, "8000", config.caip2Network);
console.log("request #2 -> cumulative maxClaimable:", voucher2.maxClaimableAmount);

const voucher3 = await signVoucher(buyerSigner, channelId, "12000", config.caip2Network);
console.log("request #3 -> cumulative maxClaimable:", voucher3.maxClaimableAmount);
console.log("\n✅ 3 requests accumulated in one channel, zero on-chain transactions so far.");

request #2 -> cumulative maxClaimable: 8000
request #3 -> cumulative maxClaimable: 12000

✅ 3 requests accumulated in one channel, zero on-chain transactions so far.


## 8b. Merchant approves the fee token for fee collection (Phase 3)

The fee is charged in the channel's token: with `TOKEN = "EURC"` the merchant approves **EURC**.

As of Phase 3 (`x402_facilitator/FEE_MODEL_PLAN.md`), `claim`/`settle` — the payload types that
actually pay out to the receiver — charge the same flat fee `exact` does, gated by the same USDC
allowance check (`checkMerchantAllowance` in `x402_fee.ts`). The manual recipient whitelist this
scheme originally ran against is retired. `deposit`/`voucher`/`refund` (sections 1-8 above) stay
free and ungated — only section 9's claim is affected.

Without an approval, section 9 will fail with `insufficient_fee_allowance` instead of succeeding.
This mirrors `x402_exact.ipynb`'s Step 3 for the `exact` scheme — same allowance,
same facilitator address, same recommended amount (1 USDC ≈ 100 settlements at the 0.01 USDC
default fee, deliberately small since the spender is a hot wallet).

Needs `NFT_WALLET_PRIVATE_KEY` in `../.env` (loaded in cell 1 as `recipientAccount`) — the private
key for `PAY_TO_ADDRESS`, since that's the address whose allowance section 9's claim draws on.

In [11]:
import { createWalletClient } from "npm:viem@2";

const ERC20_ALLOWANCE_ABI = [
    { name: "allowance", type: "function", stateMutability: "view",
      inputs: [{ name: "owner", type: "address" }, { name: "spender", type: "address" }],
      outputs: [{ name: "", type: "uint256" }] },
    { name: "approve", type: "function", stateMutability: "nonpayable",
      inputs: [{ name: "spender", type: "address" }, { name: "value", type: "uint256" }],
      outputs: [{ name: "", type: "bool" }] },
] as const;

// 1 token ≈ 100 settlements at the facilitator's 0.01 default fee (charged in the channel's token) — same recommendation
// x402_exact.ipynb uses for the exact scheme (FEE_MODEL_PLAN.md Phase 5).
const FEE_APPROVAL_AMOUNT = 1_000_000n;

if (!recipientAccount) {
    console.warn("⏭️  Skipped — NFT_WALLET_PRIVATE_KEY not set in ../.env, no key to sign approve().");
    console.warn("    Section 9's claim will fail with insufficient_fee_allowance without it.");
} else if (!FACILITATOR_ADDRESS) {
    console.warn("⏭️  Skipped — FACILITATOR_WALLET_PUBLIC_KEY not set in ../.env.");
} else {
    const currentAllowance = await publicClient.readContract({
        address: tokenAddress, abi: ERC20_ALLOWANCE_ABI, functionName: "allowance",
        args: [recipientAccount.address, FACILITATOR_ADDRESS as `0x${string}`],
    });
    console.log(`💰 Current allowance: ${formatUnits(currentAllowance, 6)} ${TOKEN}`);

    if (currentAllowance >= FEE_APPROVAL_AMOUNT) {
        console.log("   ✅ Already sufficient — skipping approve() tx.");
    } else {
        const recipientWalletClient = createWalletClient({
            account: recipientAccount, chain: config.chain, transport: http(config.rpcUrl),
        });
        console.log(`📝 Submitting approve(${FACILITATOR_ADDRESS}, ${formatUnits(FEE_APPROVAL_AMOUNT, 6)} ${TOKEN})...`);
        const approveTx = await recipientWalletClient.writeContract({
            address: tokenAddress, abi: ERC20_ALLOWANCE_ABI, functionName: "approve",
            args: [FACILITATOR_ADDRESS as `0x${string}`, FEE_APPROVAL_AMOUNT],
        });
        await publicClient.waitForTransactionReceipt({ hash: approveTx });
        console.log(`   ✅ Approved. Tx: ${config.explorer}/tx/${approveTx}`);
    }
}

💰 Current allowance: 0.98 EURC
📝 Submitting approve(0x3F8d2Fb6fEA24E70155bC61471936F3c9C30c206, 1 EURC)...
   ✅ Approved. Tx: https://sepolia.basescan.org/tx/0x316fa4abf1d6413c4a710076dfe31429c5678a53eda5be6cd165cea21ad32de1


## 9. Claim the accumulated amount in ONE on-chain transaction

This is the batch-settlement payoff: settle the 3 accumulated off-chain requests with a single
claim instead of 3 separate on-chain payments. A **claim** is a settlement command signed by the
`receiverAuthorizer` — in production, whatever runs the seller's claim/settle cron; here, the key
generated in cell 1.

**Two things to get right when building a claim by hand:**

**1. `totalClaimed` is a target, not a history.** In a `VoucherClaim` it is **the new cumulative
amount you are claiming *up to***, not "how much was already claimed". The contract reads
`if (vc.totalClaimed <= ch.totalClaimed) return;` — so passing `"0"` (or anything at or below the
current on-chain total) is a **silent no-op**: the transaction succeeds, no revert, and moves zero
value. Set it to the amount being claimed — here, `voucher3.maxClaimableAmount`.

**2. A claim does not pay out on its own.** `claim` moves the channel's escrow into a per-(receiver,
token) pending-payout bucket inside the contract — no ERC-20 transfer happens yet. A separate
`settle(receiver, token)` call sweeps that bucket to the receiver's wallet. So a successful claim
with an unchanged wallet balance is expected, not a bug. (The SDK's
`BatchSettlementChannelManager.claimAndSettle()` does both, as two transactions.)

`claimBatchTypes` and `BATCH_SETTLEMENT_DOMAIN` / `BATCH_SETTLEMENT_ADDRESS` — used below to build
the same EIP-712 domain the SDK signs internally — are public root exports of `@x402/evm`.

**Fee gate (Phase 3):** before the claim executes, the facilitator checks the receiver's USDC
allowance (the same check `exact` uses) and rejects with `insufficient_fee_allowance` if it's too
low — that's what section 8b sets up. A successful claim's response carries `fee` and
`extensions.facilitatorFees`, which the code below prints.

> ⚠️ Real on-chain transaction. To confirm it actually moved value, check the explorer's Logs tab
> for a `Claimed(channelId, sender, claimAmount, newTotalClaimed)` event — the facilitator's
> `success: true` alone is not proof, since `claimWithSignature` returns no value and the
> facilitator does not decode logs. Reading contract state immediately after a transaction can lag
> a few seconds on the public RPC; re-check after a moment if a read looks stale.

In [12]:
import { claimBatchTypes, BATCH_SETTLEMENT_DOMAIN, BATCH_SETTLEMENT_ADDRESS } from "npm:@x402/evm@^2.17.0";
import { getAddress } from "npm:viem@2";

const mayClaim = !USE_MAINNET || CONFIRM_MAINNET_WRITES;
if (!mayClaim) {
    console.warn("⏭️  Skipped — this claims REAL funds on mainnet.");
    console.warn("    Set CONFIRM_MAINNET_WRITES = true in the network-selection cell to arm it.");
}

if (mayClaim) {
    // totalClaimed = the NEW cumulative target to claim to (must be > onchain totalClaimed, ≤ maxClaimableAmount).
    // Here we're claiming the full accumulated voucher, so it equals maxClaimableAmount.
    const claimTarget = voucher3.maxClaimableAmount;

    // The receiver (server) authorizes claiming this channel's latest voucher. In
    // production the server (scw_js) signs this with its own receiverAuthorizer key;
    // here, the throwaway one generated in cell 1.
    const claimEip712Domain = {
        ...BATCH_SETTLEMENT_DOMAIN, chainId: config.chainId,
        verifyingContract: getAddress(BATCH_SETTLEMENT_ADDRESS),
    };
    const claimAuthorizerSignature = await receiverAuthorizer.signTypedData({
        domain: claimEip712Domain, types: claimBatchTypes, primaryType: "ClaimBatch",
        message: { claims: [{
            channelId, maxClaimableAmount: BigInt(voucher3.maxClaimableAmount), totalClaimed: BigInt(claimTarget),
        }] },
    });

    const claimPayload = {
        type: "claim",
        claims: [{
            voucher: { channel: channelConfig, maxClaimableAmount: voucher3.maxClaimableAmount },
            signature: voucher3.signature, totalClaimed: claimTarget,
        }],
        claimAuthorizerSignature,
    };
    const claimSettleRes = await fetch(SETTLE_URL, {
        method: "POST", headers: { "Content-Type": "application/json" },
        body: JSON.stringify({
            paymentPayload: { x402Version: 2, accepted: paymentRequirements, payload: claimPayload },
            paymentRequirements,
        }),
    });
    const claimResult = await claimSettleRes.json();
    console.log(`claim /settle (${claimSettleRes.status}):`, JSON.stringify(claimResult, null, 2));
    if (claimResult.transaction) {
        console.log(`🔗 ${config.explorer}/tx/${claimResult.transaction}`);
        console.log("\nCheck the explorer's Logs tab for a Claimed(channelId, sender, claimAmount, newTotalClaimed)");
        console.log("event — that's the definitive proof this specific claim moved real value, not just success:true.");
    }
    // Phase 3: a successful claim now carries a fee receipt, same shape exact uses.
    if (claimResult.extensions?.facilitatorFees) {
        const info = claimResult.extensions.facilitatorFees.info;
        console.log(`\n💸 Facilitator fee: ${info.facilitatorFeePaid} (${info.model}) — collection: ${info.collection.status}`);
        // The fee is charged in the channel's token, so the receipt must name it.
        const feeInChannelToken = (info.asset ?? "").toLowerCase().endsWith(tokenAddress.toLowerCase());
        console.log(`   Fee asset: ${info.asset} ${feeInChannelToken ? `✅ (${TOKEN})` : "❌ not the channel token"}`);
    }
    if (claimResult.errorReason === "insufficient_fee_allowance") {
        console.warn(`\n⚠️  Run section 8b to approve the facilitator's ${TOKEN} allowance, then retry this cell.`);
    }
}

claim /settle (200): {
  "success": true,
  "payer": "0x553179556FC2A39e535D65b921e01fA995E79101",
  "transaction": "0x745454f3926d14bdb3b031facbd275e07e92d89036fd8478b95fcc2c99f8d961",
  "network": "eip155:84532"
}
🔗 https://sepolia.basescan.org/tx/0x745454f3926d14bdb3b031facbd275e07e92d89036fd8478b95fcc2c99f8d961

Check the explorer's Logs tab for a Claimed(channelId, sender, claimAmount, newTotalClaimed)
event — that's the definitive proof this specific claim moved real value, not just success:true.


## Summary

What a full run through this notebook exercises:

| Section | What it proves |
|---|---|
| 1 | The facilitator advertises `batch-settlement` for this network |
| 3–4 | The buyer can build a valid deposit + first cumulative voucher |
| 5 | `/verify` accepts the signature, channel config, and EIP-712 domain |
| 7 | `/settle` escrows the deposit on-chain, returning the new `channelState` |
| 8 | Further requests are pure off-chain signatures — no transaction, no gas, no facilitator call |
| 8b–9 | The seller's fee allowance gates the claim, and one claim settles all accumulated vouchers |

**The payoff to notice:** section 8 adds requests to the channel with zero on-chain cost. Under the
`exact` scheme (see [`x402_exact.ipynb`](./x402_exact.ipynb)) each of those would have been its own
settlement transaction and its own flat fee. Here they cost one claim and one fee, no matter how
many vouchers accumulated.

**Not exercised here:** the `settle(receiver, token)` sweep that moves claimed funds from the
contract's pending bucket to the receiver's wallet (see section 9), and `refund`.

For why the fee model works the way it does — including why `deposit`/`voucher`/`refund` are free
while `claim`/`settle` are charged — see `x402_facilitator/FEE_MODEL_PLAN.md` Phase 3.